## 确定性 HJB 方程的完整详细推导

### 1. 问题设定

受控系统  
$$
\dot{x}(s) = f(x(s), u(s), s), \qquad s \in [t, t_f],
$$  
初始条件 $x(t) = x$。控制 $u(s) \in U \subseteq \mathbb{R}^m$，$U$ 为非空集合（可以是闭集）。

代价泛函（从时刻 $t$ 到终时 $t_f$）：
$$
J(t, x; u(\cdot)) = \int_t^{t_f} L(x(s), u(s), s)\,ds + \Phi(x(t_f)).
$$
目标是寻找最优控制 $u^*(\cdot)$，使 $J$ 达到最小。

---

### 2. 最优值函数 $V(x,t)$ 的定义

定义**最优值函数** $V(x,t)$：从任意状态 $x$、任意时刻 $t$ 出发，在所有允许控制下所能达到的最小代价：
$$
\boxed{V(x,t) = \inf_{u(\cdot) \in \mathcal{U}[t, t_f]} \left\{ \int_t^{t_f} L(x(s), u(s), s)\,ds + \Phi(x(t_f)) \;\bigg|\; \dot{x} = f(x,u,s),\; x(t)=x \right\}.}
$$

由定义直接得到终端条件：
$$
V(x, t_f) = \Phi(x). \tag{1}
$$

注意：这里的 $V$ 就是强化学习中的**最优状态值函数** $V^*$，它直接给出最优代价，没有先定义某个策略下的值函数再求最大/最小。

---

### 3. 动态规划原理（Bellman 最优性原理）

将整个时间区间 $[t, t_f]$ 分割为两段：
- 第一段：$[t, t+\Delta t]$，其中 $\Delta t > 0$ 充分小。
- 第二段：$[t+\Delta t, t_f]$。

Bellman 最优性原理：**无论初始状态和初始控制是什么，剩余决策必须构成从下一时刻起的最优策略。**

具体操作如下：

1. 在 $[t, t+\Delta t]$ 内，我们任意选择一个允许的控制 $v \in U$（可以假设在这极短时间内控制保持常值 $v$，因为 $\Delta t$ 很小，控制的变化对结果的影响是高阶无穷小）。

2. 在这段时间内，状态按照动力学演化：
   $$
   x(t+\Delta t) = x + \Delta x,
   $$
   具体形式后面近似。

3. 运行代价在这一小段时间内的积累为 $\int_t^{t+\Delta t} L(x(s), v, s)\,ds$。

4. 在时刻 $t+\Delta t$，状态变为 $x+\Delta x$。从此时起，我们承诺使用最优控制，因此剩余代价的最小值为 $V(x+\Delta x, t+\Delta t)$。

因此，如果我们**第一步使用控制 $v$，之后使用最优控制**，总代价为：
$$
\int_t^{t+\Delta t} L(x(s), v, s)\,ds + V(x(t+\Delta t), t+\Delta t).
$$

5. 最优控制要求第一步就选择最好的 $v$，以最小化总代价。因此值函数必须满足：
$$
\boxed{V(x,t) = \min_{v \in U} \left\{ \int_t^{t+\Delta t} L(x(s), v, s)\,ds + V(x(t+\Delta t), t+\Delta t) \right\}.} \tag{2}
$$

这就是**动态规划方程**（积分形式）。它与强化学习中的 Bellman 最优方程在结构上完全一致：
$$
V^*(s) = \max_a \left[ R(s,a) + \gamma \sum_{s'} P(s'|s,a) V^*(s') \right],
$$
只是这里是连续时间、确定性动力学、无折扣。

---

### 4. 导出 HJB 偏微分方程

现在对 (2) 式取极限 $\Delta t \to 0^+$，将各项做一阶近似展开。

#### 4.1 状态增量的近似

在 $[t, t+\Delta t]$ 内，若控制取常值 $v$，由动力学方程有：
$$
x(t+\Delta t) = x + \int_t^{t+\Delta t} f(x(s), v, s)\,ds.
$$
由于区间很短，$f$ 连续，用积分中值定理得：
$$
x(t+\Delta t) = x + f(x, v, t)\,\Delta t + o(\Delta t). \tag{3}
$$
记 $\Delta x = f(x, v, t)\Delta t + o(\Delta t)$.

#### 4.2 积分项的近似

同理，运行代价的积分近似为：
$$
\int_t^{t+\Delta t} L(x(s), v, s)\,ds = L(x, v, t)\,\Delta t + o(\Delta t). \tag{4}
$$

#### 4.3 值函数的泰勒展开

假设值函数 $V(x,t)$ 充分光滑（对 $x$ 和 $t$ 连续可微），将 $V(x+\Delta x, t+\Delta t)$ 在 $(x,t)$ 处做一阶泰勒展开：
$$
V(x+\Delta x, t+\Delta t) = V(x,t) + \frac{\partial V}{\partial x}(x,t) \cdot \Delta x + \frac{\partial V}{\partial t}(x,t)\,\Delta t + o(\Delta t). \tag{5}
$$
将 (3) 的 $\Delta x$ 代入：
$$
\begin{aligned}
V(x+\Delta x, t+\Delta t) &= V(x,t) + \frac{\partial V}{\partial x}(x,t) \cdot \big( f(x,v,t)\Delta t + o(\Delta t) \big) + \frac{\partial V}{\partial t}(x,t)\,\Delta t + o(\Delta t) \\
&= V(x,t) + \frac{\partial V}{\partial x} f(x,v,t)\,\Delta t + \frac{\partial V}{\partial t}\,\Delta t + o(\Delta t). \tag{6}
\end{aligned}
$$

#### 4.4 代入动态规划方程并取极限

将 (4) 和 (6) 代入 (2)：
$$
V(x,t) = \min_{v \in U} \Big\{ L(x,v,t)\Delta t + V(x,t) + \frac{\partial V}{\partial x}f(x,v,t)\Delta t + \frac{\partial V}{\partial t}\Delta t + o(\Delta t) \Big\}.
$$

两边消去 $V(x,t)$：
$$
0 = \min_{v \in U} \Big\{ L(x,v,t)\Delta t + \frac{\partial V}{\partial x}f(x,v,t)\Delta t + \frac{\partial V}{\partial t}\Delta t + o(\Delta t) \Big\}.
$$

因为 $\Delta t > 0$，可以除以 $\Delta t$：
$$
0 = \min_{v \in U} \Big\{ L(x,v,t) + \frac{\partial V}{\partial x}f(x,v,t) + \frac{\partial V}{\partial t} + \frac{o(\Delta t)}{\Delta t} \Big\}.
$$

令 $\Delta t \to 0^+$，高阶项 $\frac{o(\Delta t)}{\Delta t} \to 0$，得：
$$
0 = \min_{v \in U} \left\{ L(x,v,t) + \frac{\partial V}{\partial x}(x,t) f(x,v,t) \right\} + \frac{\partial V}{\partial t}(x,t).
$$

移项，得到**哈密顿-雅可比-贝尔曼（HJB）方程**：
$$
\boxed{-\frac{\partial V}{\partial t}(x,t) = \min_{v \in U} \left\{ L(x,v,t) + \frac{\partial V}{\partial x}(x,t) \cdot f(x,v,t) \right\}.} \tag{7}
$$

终端条件由 (1) 给出：$V(x, t_f) = \Phi(x)$.

这就是确定性最优控制的 HJB 方程。它是一个一阶非线性偏微分方程，解出 $V(x,t)$ 后，最优控制可以通过对每个 $(x,t)$ 求解右侧的最小化问题得到。

---

### 5. 动作值函数 $Q(x,u,t)$ 的引入

定义**动作值函数**：
$$
\boxed{Q(x, u, t) = L(x, u, t) + \frac{\partial V}{\partial x}(x,t) \cdot f(x, u, t).}
$$
$Q$ 的含义是：在状态 $x$、时刻 $t$ 采取控制 $u$ 时，所产生的“瞬时代价 + 状态变化导致的值函数变化率”，它直接衡量该动作的“好坏”。

利用 $Q$，HJB 方程 (7) 可写成极其简洁的形式：
$$
\boxed{-\frac{\partial V}{\partial t}(x,t) = \min_{u \in U} Q(x,u,t).} \tag{8}
$$

而最优控制（反馈形式）为：
$$
u^*(x,t) = \arg\min_{u \in U} Q(x,u,t).
$$

---

### 6. 与庞特里亚金最小值原理（PMP）的联系

定义哈密顿量：
$$
H(x, u, \lambda, t) = L(x,u,t) + \lambda^\top f(x,u,t).
$$
在 HJB 中，令 $\lambda = \left(\frac{\partial V}{\partial x}(x,t)\right)^\top$，则
$$
Q(x,u,t) = H(x, u, \nabla_x V^\top, t),
$$
且 HJB 方程变为：
$$
-\frac{\partial V}{\partial t} = \min_{u \in U} H(x, u, \nabla_x V^\top, t).
$$

沿着最优轨迹 $x^*(\cdot)$，定义 $\lambda(t) = \nabla_x V(x^*(t), t)^\top$，可以证明 $\lambda(t)$ 满足庞特里亚金的伴随方程 $\dot{\lambda} = -\frac{\partial H}{\partial x}$，并且最小值条件与 PMP 完全相同。

- **HJB**：全局的充分必要条件（需要解 PDE）。
- **PMP**：沿单条轨迹的必要条件（转化为两点边值 ODE 问题）。

---

### 7. 推导总结

| 步骤 | 内容 |
|------|------|
| 定义值函数 | $V(x,t) = \min_{u} \{\text{总代价}\}$，终端 $V=\Phi$ |
| Bellman 原理 | $V(x,t) = \min_{v} \{ \int_t^{t+\Delta t} L ds + V(x+\Delta x, t+\Delta t) \}$ |
| 一阶近似 | $\Delta x \approx f\Delta t$，积分 $\approx L\Delta t$，值函数泰勒展开 |
| 取极限 | 除以 $\Delta t$，令 $\Delta t \to 0$，高阶项消失 |
| HJB 方程 | $-\frac{\partial V}{\partial t} = \min_{u} \{ L + V_x f \}$ |
| 动作值函数 | $Q = L + V_x f$，HJB 为 $-\frac{\partial V}{\partial t} = \min_u Q$ |
| 最优控制 | $u^*(x,t) = \arg\min_u Q$ |
| 与 PMP 联系 | $\lambda = V_x^\top$ 满足伴随方程，PMP 沿轨迹成立 |

整个推导的核心在于：将全局最优问题通过动态规划原理转化为局部瞬时决策问题，然后取时间步长趋于零的极限，从而得到描述最优值函数的偏微分方程。这一过程与离散时间强化学习中从贝尔曼方程到最优策略的方法一脉相承。


---

## 随机HJB方程的完整详细推导

### 1. 问题设定

考虑一个由伊藤随机微分方程（SDE）描述的控制系统：

$$
dX(s) = f(X(s), u(s), s)\,ds + \sigma(X(s), u(s), s)\,dW(s), \quad s \in [t, t_f],
$$

初始条件 $X(t) = x$（确定值）。其中：
- $X(s) \in \mathbb{R}^n$：状态向量。
- $u(s) \in U \subseteq \mathbb{R}^m$：控制输入，$U$ 是非空集合（可以是闭集）。
- $W(s) \in \mathbb{R}^d$：$d$ 维标准布朗运动，其分量相互独立。
- $f: \mathbb{R}^n \times \mathbb{R}^m \times \mathbb{R} \to \mathbb{R}^n$：漂移系数。
- $\sigma: \mathbb{R}^n \times \mathbb{R}^m \times \mathbb{R} \to \mathbb{R}^{n \times d}$：扩散系数。

代价函数定义为期望值：

$$
J(t, x; u(\cdot)) = \mathbb{E}\left[ \int_t^{t_f} L(X(s), u(s), s)\,ds + \Phi(X(t_f)) \;\bigg|\; X(t) = x \right],
$$

其中 $L$ 是运行代价，$\Phi$ 是终端代价。目标是最小化 $J$。

---

### 2. 最优值函数的定义

定义**最优值函数** $V(x,t)$ 为从状态 $x$、时刻 $t$ 出发，在所有允许控制下所能达到的最小期望代价：

$$
\boxed{V(x,t) = \inf_{u(\cdot)} \mathbb{E}\left[ \int_t^{t_f} L(X(s), u(s), s)\,ds + \Phi(X(t_f)) \;\bigg|\; X(t) = x \right].}
$$

由定义，在终端时刻 $t = t_f$ 没有剩余时间，只有终端代价，因此：

$$
V(x, t_f) = \Phi(x). \tag{1}
$$

---

### 3. Bellman最优性原理（随机版本）

将时间区间 $[t, t_f]$ 分成两段：$[t, t+\Delta t]$ 和 $[t+\Delta t, t_f]$，其中 $\Delta t > 0$ 充分小。

在第一步 $[t, t+\Delta t]$ 内，我们任取一个常值控制 $v \in U$。由于系统的随机性，状态 $X(t+\Delta t)$ 是随机变量。从时刻 $t+\Delta t$ 起，假设我们采用最优控制，则后续的期望最小代价为 $V(X(t+\Delta t), t+\Delta t)$。

于是，若第一步采用 $v$，则从 $t$ 出发的总期望代价为：

$$
\mathbb{E}\left[ \int_t^{t+\Delta t} L(X(s), v, s)\,ds + V(X(t+\Delta t), t+\Delta t) \;\bigg|\; X(t)=x \right].
$$

要获得最优值 $V(x,t)$，我们必须在第一步选择最好的 $v$，即对所有 $v$ 取最小：

$$
\boxed{V(x,t) = \min_{v \in U} \mathbb{E}\left[ \int_t^{t+\Delta t} L(X(s), v, s)\,ds + V(X(t+\Delta t), t+\Delta t) \;\bigg|\; X(t)=x \right].} \tag{2}
$$

这就是**随机Bellman方程**（积分形式），也是推导的起点。它完全类似于确定性情况，只是多出了条件期望。

---

### 4. 使用伊藤引理展开值函数

我们需要处理 (2) 式中的条件期望。关键在于计算 $\mathbb{E}[V(X(t+\Delta t), t+\Delta t) \mid X(t)=x]$。

#### 4.1 状态增量

在 $[t, t+\Delta t]$ 内，控制保持常值 $v$，SDE 可以近似离散化为：

$$
\Delta X := X(t+\Delta t) - x = f(x, v, t) \Delta t + \sigma(x, v, t) \Delta W + o(\Delta t),
$$

其中 $\Delta W = W(t+\Delta t) - W(t) \sim \mathcal{N}(0, \Delta t I_d)$。注意 $\Delta W \in \mathbb{R}^d$，其分量独立，且满足：

$$
\mathbb{E}[\Delta W] = 0, \qquad \mathbb{E}[\Delta W \Delta W^\top] = I_d \Delta t.
$$

#### 4.2 伊藤引理（高维形式）

值函数 $V(t, X)$ 是时间和状态的光滑函数。由于 $X$ 是伊藤过程，我们不能用普通微积分展开，而必须使用**伊藤引理**。

设 $X$ 满足 $dX = \mu dt + \Sigma dW$，则对光滑函数 $g(t, X)$，有

$$
dg = \left( \frac{\partial g}{\partial t} + (\nabla_X g)^\top \mu + \frac{1}{2} \operatorname{Tr}\left( \Sigma \Sigma^\top \nabla_X^2 g \right) \right)dt + (\nabla_X g)^\top \Sigma \, dW.
$$

应用到值函数 $V(t, X)$，此时漂移为 $f$，扩散为 $\sigma$，所以：

$$
\begin{aligned}
dV(t, X) &= \left( \frac{\partial V}{\partial t} + (\nabla_X V)^\top f + \frac{1}{2} \operatorname{Tr}\!\left( \sigma \sigma^\top \nabla_X^2 V \right) \right)dt \\
&\quad + (\nabla_X V)^\top \sigma \, dW.
\end{aligned}
$$

这里：
- $\nabla_X V$ 是 $V$ 对 $X$ 的梯度（列向量）。
- $\nabla_X^2 V$ 是 Hessian 矩阵。
- $\sigma \sigma^\top$ 是 $n \times n$ 的对称半正定矩阵，代表噪声引起的状态协方差。

#### 4.3 计算条件期望

将 $dV$ 的表达式从 $t$ 到 $t+\Delta t$ 积分：

$$
V(X(t+\Delta t), t+\Delta t) = V(x,t) + \int_t^{t+\Delta t} dV(s).
$$

取条件期望 $\mathbb{E}[\cdot \mid X(t)=x]$。由于伊藤积分 $\int (\nabla_X V)^\top \sigma \, dW$ 的期望为零（鞅性质），我们只需处理漂移项的积分：

$$
\begin{aligned}
&\mathbb{E}[V(X(t+\Delta t), t+\Delta t) \mid X(t)=x] \\
&= V(x,t) + \mathbb{E}\left[ \int_t^{t+\Delta t} \left( \frac{\partial V}{\partial t} + (\nabla_X V)^\top f + \frac{1}{2} \operatorname{Tr}(\sigma \sigma^\top \nabla_X^2 V) \right) ds \;\bigg|\; X(t)=x \right].
\end{aligned}
$$

由于在极短的 $\Delta t$ 区间内，积分可近似为被积函数在 $t$ 处取值乘以 $\Delta t$（加上高阶项）：

$$
\begin{aligned}
&= V(x,t) + \left( \frac{\partial V}{\partial t} + (\nabla_X V)^\top f(x,v,t) + \frac{1}{2} \operatorname{Tr}\!\left( \sigma(x,v,t) \sigma(x,v,t)^\top \nabla_X^2 V(x,t) \right) \right) \Delta t \\
&\quad + o(\Delta t). \tag{3}
\end{aligned}
$$

**这就是值函数展开后的条件期望**。相比确定性情况，多出了 **$\frac{1}{2} \operatorname{Tr}(\sigma\sigma^\top \nabla_X^2 V) \Delta t$** 这一项，它直接来源于伊藤引理的二阶扩散项。

---

### 5. 代入Bellman方程并取极限

现在回到 (2) 式。先将运行代价的积分也做近似：

$$
\int_t^{t+\Delta t} L(X(s), v, s) ds = L(x, v, t) \Delta t + o(\Delta t).
$$

将上述积分近似和 (3) 式代入 (2)：

$$
\begin{aligned}
V(x,t) &= \min_{v \in U} \Bigg\{ L(x,v,t)\Delta t + V(x,t) \\
&\qquad + \left( \frac{\partial V}{\partial t} + (\nabla_X V)^\top f(x,v,t) + \frac{1}{2} \operatorname{Tr}\!\left( \sigma\sigma^\top \nabla_X^2 V \right) \right) \Delta t \\
&\qquad + o(\Delta t) \Bigg\}.
\end{aligned}
$$

两边消去 $V(x,t)$，得：

$$
0 = \min_{v \in U} \Bigg\{ L(x,v,t)\Delta t + \left( \frac{\partial V}{\partial t} + (\nabla_X V)^\top f(x,v,t) + \frac{1}{2} \operatorname{Tr}\!\left( \sigma\sigma^\top \nabla_X^2 V \right) \right) \Delta t + o(\Delta t) \Bigg\}.
$$

除以 $\Delta t$（$\Delta t > 0$），并令 $\Delta t \to 0^+$，高阶项 $o(\Delta t)/\Delta t \to 0$，得到：

$$
0 = \min_{v \in U} \left\{ L(x,v,t) + (\nabla_X V)^\top f(x,v,t) + \frac{1}{2} \operatorname{Tr}\!\left( \sigma\sigma^\top \nabla_X^2 V \right) + \frac{\partial V}{\partial t} \right\}.
$$

移项，得到**随机HJB方程**：

$$
\boxed{-\frac{\partial V}{\partial t}(x,t) = \min_{v \in U} \left\{ L(x,v,t) + (\nabla_X V)^\top f(x,v,t) + \frac{1}{2} \operatorname{Tr}\!\left( \sigma(x,v,t) \sigma(x,v,t)^\top \nabla_X^2 V(x,t) \right) \right\}.}
$$

终端条件：$V(x, t_f) = \Phi(x)$.

---

### 6. 动作值函数 $Q$ 与最优控制

定义**动作值函数**（$Q$-函数）为上述最小化目标：

$$
\boxed{Q(x, u, t) = L(x, u, t) + (\nabla_X V)^\top f(x, u, t) + \frac{1}{2} \operatorname{Tr}\!\left( \sigma(x, u, t) \sigma(x, u, t)^\top \nabla_X^2 V(x,t) \right).}
$$

则随机HJB方程可写作：

$$
-\frac{\partial V}{\partial t} = \min_{u \in U} Q(x, u, t).
$$

最优反馈控制为：

$$
u^*(x, t) = \arg\min_{u \in U} Q(x, u, t).
$$

---

### 7. 与确定性HJB的对比

| 方面 | 确定性 HJB | 随机 HJB |
|------|-----------|----------|
| 状态方程 | $\dot{x} = f(x,u)$ | $dX = f\,dt + \sigma\,dW$ |
| 值函数展开工具 | 普通泰勒展开（一阶） | **伊藤引理**（保留二阶） |
| 优化目标 | $L + V_x^\top f$ | $L + V_x^\top f + \frac{1}{2}\operatorname{Tr}(\sigma\sigma^\top V_{xx})$ |
| 新增项来源 | — | 布朗运动二次变差 $(dW)^2 = dt$ |
| 控制含义 | 仅优化确定性漂移 | 同时优化漂移和扩散程度 |

**核心差异**：随机HJB中的迹项 $\frac{1}{2}\operatorname{Tr}(\sigma\sigma^\top \nabla_X^2 V)$ 代表了噪声扩散对预期代价的影响。最优控制需要在“朝好的方向移动”与“控制不确定性（风险）”之间权衡。如果 $\sigma$ 本身依赖于控制 $u$，那么控制甚至可以通过调节噪声强度来优化性能。

---

### 8. 与随机最大值原理的联系

若值函数 $V$ 足够光滑，定义沿最优轨迹的伴随过程：
$$
\lambda(t) = \nabla_X V(X^*(t), t), \quad \beta(t) = \nabla_X^2 V(X^*(t), t) \, \sigma(X^*(t), u^*(t), t).
$$
可以证明 $\lambda(t)$ 满足一个倒向随机微分方程（BSDE），并且随机HJB中的最小化目标等于随机哈密顿量 $H = L + \lambda^\top f + \operatorname{Tr}(\beta^\top \sigma)$，从而与**随机庞特里亚金最大值原理**等价。HJB提供全局充要条件，而随机PMP提供沿单条轨迹的必要条件。

---

**总结**：随机HJB方程是将Bellman最优性原理与伊藤随机分析相结合的自然产物。其推导核心在于用伊藤引理展开值函数，保留由于布朗运动二次变差非零而产生的二阶扩散项，最终得到带有迹项的偏微分方程。这一方程全面刻画了受噪声影响的最优决策。